In [2]:
import json
import numpy as np
import pandas as pd

In [3]:
n_splits=3
seed=90051
# Load the shared subset metadata and outer CV splits
df = pd.read_csv("expw_subset_metadata_comp90051.csv")
correct_facial_class = df["correct_facial_class"].to_numpy()
with open("outer_10-fold_CV_splits.json", "r") as f:
    outer_splits = json.load(f)

In [4]:
def k_fold_CV_function(correct_facial_class,n_splits,seed):
    start = 0
    folds_list = [[] for _ in range(n_splits)]
    random_generator = np.random.default_rng(seed)
    correct_class_narray = np.asarray(correct_facial_class)
    for facial_class in np.unique(correct_class_narray):
        f_class_idxs = np.where(correct_class_narray == facial_class)[0]
        random_generator.shuffle(f_class_idxs)
        for i in range(len(f_class_idxs)):
            fold_num = (start + i) % n_splits
            folds_list[fold_num].append(int(f_class_idxs[i]))
        start = (start + len(f_class_idxs)) % n_splits
    result = []
    for fold in folds_list:
        result.append(np.array(sorted(fold), dtype=int))
    return result

In [20]:
# Generate inner 3-fold CV for each outer fold
inner_splits = {}
start=1
for outer_fold in outer_splits:
    fold_id = outer_fold["fold number"]
    outer_train_indexes = np.array(outer_fold["train indexes"],dtype=int)
    outer_train_facial_class = correct_facial_class[outer_train_indexes]
    inner_val_folds = k_fold_CV_function(outer_train_facial_class, n_splits, seed)
    inner_splits[fold_id] = []
    all_positions = np.arange(len(outer_train_indexes))
    for inner_fold_id, inner_val_positions in enumerate(inner_val_folds,start):
        inner_train_positions = np.setdiff1d(all_positions,inner_val_positions)
        train_idx = outer_train_indexes[inner_train_positions]
        val_idx = outer_train_indexes[inner_val_positions]
        inner_splits[fold_id].append({"inner_fold": inner_fold_id,"train_indexes": train_idx.tolist(),"val_indexes": val_idx.tolist()})
    print(f"Outer Fold {fold_id}: "f"{len(inner_splits[fold_id])} inner folds generated")

Outer Fold 1: 3 inner folds generated
Outer Fold 2: 3 inner folds generated
Outer Fold 3: 3 inner folds generated
Outer Fold 4: 3 inner folds generated
Outer Fold 5: 3 inner folds generated
Outer Fold 6: 3 inner folds generated
Outer Fold 7: 3 inner folds generated
Outer Fold 8: 3 inner folds generated
Outer Fold 9: 3 inner folds generated
Outer Fold 10: 3 inner folds generated
